# Semantic search over documentation with LanceDB

This notebook builds a searchable index of the [FastAPI documentation](https://fastapi.tiangolo.com) with [LanceDB](https://docs.lancedb.com). It downloads the docs, splits each page into sections, stores the sections in a LanceDB table that embeds text automatically, and answers natural-language questions with vector search.

**Before you start**

- Python 3.12 (tested with 3.12.3 on Ubuntu 24.04 under WSL)
- Git, used to download the docs
- The packages in [`requirements.txt`](requirements.txt), installed in a virtual environment as described in the [README](README.md)
- No API keys: embeddings run locally with the `all-MiniLM-L6-v2` model

The first run downloads the FastAPI docs from GitHub and the embedding model from Hugging Face, so it needs an internet connection. Later runs reuse both.

**Data source and license:** The FastAPI documentation is © Sebastián Ramírez and contributors, used under the [MIT License](https://github.com/fastapi/fastapi/blob/master/LICENSE). The notebook downloads it when it runs; this repository doesn't include a copy.

## Setup

Settings used throughout the notebook. `FASTAPI_REF` pins the docs to one FastAPI release, so results are reproducible.

In [1]:
import re
import subprocess
from pathlib import Path

import lancedb
import pandas as pd
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

FASTAPI_REF = "0.159.0"          # FastAPI release whose docs we index
DATA_DIR = Path("data")          # Everything this notebook downloads or creates
DOCS_REPO = DATA_DIR / "fastapi"
DOCS_ROOT = DOCS_REPO / "docs" / "en" / "docs"
DB_PATH = DATA_DIR / "lancedb"
TABLE_NAME = "fastapi_docs"
MODEL_NAME = "all-MiniLM-L6-v2"

print("LanceDB version:", lancedb.__version__)

LanceDB version: 0.40.0


## Step 1: Download the docs

A sparse, shallow Git clone fetches only the English docs folder at the pinned release, instead of the whole repository and its history.

In [2]:
def download_docs(ref: str = FASTAPI_REF) -> None:
    if DOCS_ROOT.exists():
        print(f"Docs already downloaded to {DOCS_ROOT}")
        return
    DATA_DIR.mkdir(exist_ok=True)
    subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1", "--branch", ref,
         "--filter=blob:none", "--sparse",
         "https://github.com/fastapi/fastapi.git", str(DOCS_REPO)],
        check=True,
    )
    subprocess.run(["git", "-C", str(DOCS_REPO), "sparse-checkout", "set", "docs/en/docs"], check=True)
    print(f"Downloaded FastAPI {ref} docs to {DOCS_ROOT}")

download_docs()
pages = sorted(DOCS_ROOT.rglob("*.md"))
print(f"{len(pages)} Markdown pages found")

Note: switching to '68cc4cbb2e0600841b0d6bdd1cc29038b54a00e6'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



Downloaded FastAPI 0.159.0 docs to data/fastapi/docs/en/docs
156 Markdown pages found


## Step 2: Clean and split the pages

Search works better on focused passages than on whole pages, so each page is split at its `##` headings. Very long sections are split again at paragraph breaks, because the embedding model reads only about the first 250 words of a passage.

The cleaning step removes markup that only means something to the docs site's build tool: heading anchors such as `{ #query-parameters }`, code-include directives such as `{* ../../docs_src/... *}`, admonition fences (`///`), and HTML tags. Pages with little prose to search, such as the release notes and the generated API reference stubs, are skipped.

In [3]:
SKIP_PAGES = {
    "release-notes.md", "_llm-test.md", "translation-banner.md",
    "fastapi-people.md", "management.md", "management-tasks.md", "newsletter.md",
}
SKIP_FOLDERS = {"reference"}      # Generated API reference stubs
MAX_WORDS = 250                   # Roughly what the embedding model reads

DIRECTIVE = re.compile(r"^\s*(\{\*.*\*\}|\{!.*!\}|:::|////?)")
ANCHOR = re.compile(r"\s*\{\s*#[^}]*\}\s*$")
HTML_TAG = re.compile(r"<[^>]+>")


def clean_lines(text: str) -> list[str]:
    lines = []
    for line in text.splitlines():
        if DIRECTIVE.match(line):
            continue
        line = ANCHOR.sub("", line)
        line = HTML_TAG.sub("", line)
        lines.append(line)
    return lines


def page_url(path: Path) -> str:
    rel = path.relative_to(DOCS_ROOT).with_suffix("")
    parts = [] if rel.name == "index" and rel.parent == Path(".") else list(rel.parts)
    if parts and parts[-1] == "index":
        parts = parts[:-1]
    return "https://fastapi.tiangolo.com/" + "".join(p + "/" for p in parts)


def split_long(text: str, max_words: int = MAX_WORDS) -> list[str]:
    if len(text.split()) <= max_words:
        return [text]
    parts, current = [], []
    for para in re.split(r"\n\s*\n", text):
        if current and len(" ".join(current + [para]).split()) > max_words:
            parts.append("\n\n".join(current))
            current = []
        current.append(para)
    if current:
        parts.append("\n\n".join(current))
    return parts


def chunk_page(path: Path) -> list[dict]:
    rel = path.relative_to(DOCS_ROOT)
    section = rel.parts[0] if len(rel.parts) > 1 else "general"
    title, heading, body, in_code = path.stem, "Introduction", [], False
    sections = []
    for line in clean_lines(path.read_text(encoding="utf-8")):
        if line.lstrip().startswith("```"):
            in_code = not in_code
        if not in_code and line.startswith("# "):
            title = line[2:].strip()
            continue
        if not in_code and line.startswith("## "):
            sections.append((heading, body))
            heading, body = line[3:].strip(), []
            continue
        body.append(line)
    sections.append((heading, body))

    chunks = []
    for heading, lines in sections:
        text = "\n".join(lines).strip()
        if len(text.split()) < 15:            # Skip empty or near-empty sections
            continue
        for i, part in enumerate(split_long(text)):
            chunks.append({
                "id": f"{rel.as_posix()}#{len(chunks)}",
                "page_title": title,
                "heading": heading,
                "section": section,
                "url": page_url(path),
                "word_count": len(part.split()),
                "text": f"{title}: {heading}\n\n{part}",
            })
    return chunks


chunks = []
for path in pages:
    rel = path.relative_to(DOCS_ROOT)
    if path.name in SKIP_PAGES or rel.parts[0] in SKIP_FOLDERS:
        continue
    chunks.extend(chunk_page(path))

df = pd.DataFrame(chunks)
print(f"{len(df)} chunks from {df['url'].nunique()} pages")
df.groupby("section").size().sort_values(ascending=False)

814 chunks from 124 pages


section
tutorial      375
advanced      193
general       107
deployment     84
how-to         54
learn           1
dtype: int64

In [4]:
df[["page_title", "heading", "section", "word_count", "url"]].head(10)

,page_title,heading,section,word_count,url
0,Additional Responses in OpenAPI,Introduction,advanced,72,https://fastapi.tiangolo.com/advanced/addition...
1,Additional Responses in OpenAPI,Additional Response with `model`,advanced,238,https://fastapi.tiangolo.com/advanced/addition...
2,Additional Responses in OpenAPI,Additional Response with `model`,advanced,196,https://fastapi.tiangolo.com/advanced/addition...
3,Additional Responses in OpenAPI,Additional media types for the main response,advanced,115,https://fastapi.tiangolo.com/advanced/addition...
4,Additional Responses in OpenAPI,Combining information,advanced,123,https://fastapi.tiangolo.com/advanced/addition...
5,Additional Responses in OpenAPI,Combine predefined responses and custom ones,advanced,122,https://fastapi.tiangolo.com/advanced/addition...
6,Additional Responses in OpenAPI,More information about OpenAPI responses,advanced,65,https://fastapi.tiangolo.com/advanced/addition...
7,Additional Status Codes,Introduction,advanced,39,https://fastapi.tiangolo.com/advanced/addition...
8,Additional Status Codes,Additional status codes,advanced,187,https://fastapi.tiangolo.com/advanced/addition...
9,Additional Status Codes,OpenAPI and API docs,advanced,46,https://fastapi.tiangolo.com/advanced/addition...


## Step 3: Create the table with automatic embeddings

LanceDB's embedding registry ties an embedding model to the table's schema. The `text` field is marked as the source and `vector` as the destination, so LanceDB embeds text automatically when rows are added and when you search with a string. You never call the model yourself.

The first time this cell runs, it downloads the model (about 90 MB). Embedding all the chunks takes a minute or two on a laptop CPU.

In [5]:
embedder = get_registry().get("sentence-transformers").create(name=MODEL_NAME, device="cpu")


class DocChunk(LanceModel):
    id: str
    page_title: str
    heading: str
    section: str
    url: str
    word_count: int
    text: str = embedder.SourceField()
    vector: Vector(embedder.ndims()) = embedder.VectorField()


db = lancedb.connect(DB_PATH)
table = db.create_table(TABLE_NAME, schema=DocChunk, mode="overwrite")
table.add(df.to_dict("records"))
print(f"Table '{TABLE_NAME}' has {table.count_rows()} rows; vectors have {embedder.ndims()} dimensions")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[2026-10-21T23:20:53Z WARN  lance::dataset::write::insert] No existing dataset at /home/runner/work/agent-ready-docs/agent-ready-docs/case-studies/lancedb/examples/data/lancedb/fastapi_docs.lance, it will be created


Table 'fastapi_docs' has 814 rows; vectors have 384 dimensions


## Step 4: Ask questions with vector search

Vector search compares the meaning of the question with the meaning of each chunk, so it finds relevant passages even when they use different words. A smaller `_distance` means a closer match.

In [6]:
def ask(question: str, k: int = 5) -> pd.DataFrame:
    return (
        table.search(question)
        .limit(k)
        .select(["page_title", "heading", "url", "_distance"])
        .to_pandas()
    )


ask("How do I make a query parameter optional?")

,page_title,heading,url,_distance
0,Query Parameters and String Validations,Default values,https://fastapi.tiangolo.com/tutorial/query-pa...,0.606595
1,Query Parameters,Optional parameters,https://fastapi.tiangolo.com/tutorial/query-pa...,0.705053
2,Query Parameters and String Validations,Required parameters,https://fastapi.tiangolo.com/tutorial/query-pa...,0.777589
3,Query Parameters and String Validations,Deprecating parameters,https://fastapi.tiangolo.com/tutorial/query-pa...,0.850110
4,Query Parameters,Required query parameters,https://fastapi.tiangolo.com/tutorial/query-pa...,0.869601


In [7]:
ask("How can I run code after returning a response?")

,page_title,heading,url,_distance
0,Response - Change Status Code,Introduction,https://fastapi.tiangolo.com/advanced/response...,0.856890
1,Response - Change Status Code,Use case,https://fastapi.tiangolo.com/advanced/response...,0.902061
2,"Custom Response - HTML, Stream, File, others",Available responses,https://fastapi.tiangolo.com/advanced/custom-r...,0.949753
3,Response Status Code,Changing the default,https://fastapi.tiangolo.com/tutorial/response...,0.979274
4,Return a Response Directly,Returning a custom `Response`,https://fastapi.tiangolo.com/advanced/response...,0.991986


The first question works well. The second one doesn't: the best answer is the [Background Tasks](https://fastapi.tiangolo.com/tutorial/background-tasks/) page, whose first sentence is "You can define background tasks to be run *after* returning a response." Vector search ranks pages about changing responses higher instead, because a small embedding model reduces each section to one overall meaning, and the rest of that section is about email notifications and file processing.

Step 6 comes back to this question to see whether other search methods do better.

## Step 5: Filter results

A `where` clause narrows a search with a SQL condition on any column. Here, the search is limited to the Tutorial section of the docs.

By default, LanceDB applies the filter **before** the vector search (a prefilter), so you always get up to `limit` matching rows. With `prefilter=False`, it applies the filter **after** finding the nearest rows (a postfilter), which can return fewer rows than you asked for, because some of the nearest rows are then filtered out.

In [8]:
question = "How do I add authentication to my API?"

prefiltered = (
    table.search(question)
    .where("section = 'tutorial'")
    .limit(5)
    .select(["page_title", "heading", "section", "_distance"])
    .to_pandas()
)
prefiltered

,page_title,heading,section,_distance
0,Security - First Steps,Introduction,tutorial,0.758450
1,Simple OAuth2 with Password and Bearer,Introduction,tutorial,0.776611
2,Simple OAuth2 with Password and Bearer,Recap,tutorial,0.825074
3,"OAuth2 with Password (and hashing), Bearer wit...",Advanced usage with `scopes`,tutorial,0.841569
4,Security - First Steps,The `password` flow,tutorial,0.877060


In [9]:
postfiltered = (
    table.search(question)
    .where("section = 'tutorial'", prefilter=False)
    .limit(5)
    .select(["page_title", "heading", "section", "_distance"])
    .to_pandas()
)
print(f"Prefilter returned {len(prefiltered)} rows; postfilter returned {len(postfiltered)} rows")
postfiltered

Prefilter returned 5 rows; postfilter returned 4 rows


,page_title,heading,section,_distance
0,Security - First Steps,Introduction,tutorial,0.758450
1,Simple OAuth2 with Password and Bearer,Introduction,tutorial,0.776611
2,Simple OAuth2 with Password and Bearer,Recap,tutorial,0.825074
3,"OAuth2 with Password (and hashing), Bearer wit...",Advanced usage with `scopes`,tutorial,0.841569


In the run saved in this notebook, the prefilter returned five rows and the postfilter returned four: one of the five rows nearest the question overall was outside the Tutorial section, so the postfilter removed it and had nothing to replace it with. Use a prefilter unless you have a specific reason not to.

## Step 6: Full-text and hybrid search

Vector search matches meaning; full-text search matches words. LanceDB supports both on the same table and can combine them into a **hybrid search**. Combining means merging two ranked lists, which is the job of a **reranker**:

- `RRFReranker` (reciprocal rank fusion) merges the lists by rank position alone. It's fast and needs no model.
- `CrossEncoderReranker` reads the question and each candidate passage together with a small model and scores how well the passage answers the question. It's slower but usually more accurate. The first run downloads its model from Hugging Face.

Full-text search needs an index on the text column first.

In [10]:
from lancedb.index import FTS
from lancedb.rerankers import CrossEncoderReranker, RRFReranker

table.create_index("text", config=FTS(), replace=True)
cross_encoder = CrossEncoderReranker()


def top_hits(query, k: int = 5) -> list[str]:
    rows = query.limit(k).to_pandas()
    return [f"{r.page_title}: {r.heading}" for r in rows.itertuples()]


def compare(question: str, k: int = 5) -> pd.DataFrame:
    return pd.DataFrame({
        "Vector": top_hits(table.search(question), k),
        "Full-text": top_hits(table.search(question, query_type="fts"), k),
        "Hybrid (RRF)": top_hits(table.search(question, query_type="hybrid").rerank(RRFReranker()), k),
        "Hybrid (cross-encoder)": top_hits(table.search(question, query_type="hybrid").rerank(cross_encoder), k),
    }, index=range(1, k + 1))

In [11]:
compare("How can I run code after returning a response?")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

,Vector,Full-text,Hybrid (RRF),Hybrid (cross-encoder)
1,Response - Change Status Code: Introduction,Middleware: Introduction,Response - Change Status Code: Use case,"Custom Response - HTML, Stream, File, others: ..."
2,Response - Change Status Code: Use case,Dependencies with yield: Early exit and `scope`,"Custom Response - HTML, Stream, File, others: ...",Additional Status Codes: Additional status codes
3,"Custom Response - HTML, Stream, File, others: ...",Additional Status Codes: Additional status codes,Response - Change Status Code: Introduction,Middleware: Introduction
4,Response Status Code: Changing the default,"Custom Response - HTML, Stream, File, others: ...",Middleware: Introduction,Response - Change Status Code: Use case
5,Return a Response Directly: Returning a custom...,Response - Change Status Code: Use case,Dependencies with yield: Early exit and `scope`,Return a Response Directly: Returning a custom...


In [12]:
compare("How do I make a query parameter optional?")

,Vector,Full-text,Hybrid (RRF),Hybrid (cross-encoder)
1,Query Parameters and String Validations: Defau...,Query Parameter Models: Optional Query Paramet...,Query Parameters and String Validations: Defau...,Query Parameters: Optional parameters
2,Query Parameters: Optional parameters,Cookie Parameter Models: Optional Cookie Param...,Query Parameters: Required query parameters,Query Parameter Models: Optional Query Paramet...
3,Query Parameters and String Validations: Requi...,Header Parameter Models: Optional Header Param...,Query Parameter Models: Optional Query Paramet...,Query Parameters: Required query parameters
4,Query Parameters and String Validations: Depre...,Query Parameters: Required query parameters,Query Parameters: Optional parameters,Cookie Parameter Models: Optional Cookie Param...
5,Query Parameters: Required query parameters,Query Parameters and String Validations: Defau...,Cookie Parameter Models: Optional Cookie Param...,Header Parameter Models: Optional Header Param...


**What the comparison shows.** With default settings, none of the four methods finds the Background Tasks page for the first question. There are two reasons, and both are general lessons about search.

1. **The word that matters was thrown away.** By default, the full-text index removes *stop words*: very common words such as "a", "the", and "after". Without "after", the question becomes "run code return response", and those words appear all over the FastAPI docs. The next cell confirms that "after" alone matches nothing.
2. **A reranker can only reorder what retrieval found.** Both hybrid methods rerank candidates from the vector and full-text searches. If neither search returns the Background Tasks page, no reranker can promote it.

The second question goes better. Every method returns relevant pages, and the cross-encoder puts the best answer, "Query Parameters: Optional parameters", first. Full-text search drifts toward the Cookie and Header parameter pages, which share the words "optional" and "parameters" but answer a different question.

In [13]:
print("Full-text matches for 'after':", len(table.search("after", query_type="fts").limit(1000).to_pandas()))

Full-text matches for 'after': 0


### Keep stop words in the full-text index

Rebuilding the index with `remove_stop_words=False` keeps "after" searchable. `with_position=True` also stores where each word appears, which enables exact-phrase search. Both make the index larger, which is the reason they're off by default.

In [14]:
table.create_index("text", config=FTS(remove_stop_words=False, with_position=True), replace=True)

print("Full-text matches for 'after':", len(table.search("after", query_type="fts").limit(1000).to_pandas()))
compare("How can I run code after returning a response?")

Full-text matches for 'after': 64


,Vector,Full-text,Hybrid (RRF),Hybrid (cross-encoder)
1,Response - Change Status Code: Introduction,Middleware: Introduction,Response - Change Status Code: Introduction,Background Tasks: Introduction
2,Response - Change Status Code: Use case,Dependencies with yield: Early exit and `scope`,Middleware: Introduction,"Custom Response - HTML, Stream, File, others: ..."
3,"Custom Response - HTML, Stream, File, others: ...",Advanced Dependencies: Dependencies with `yiel...,Response - Change Status Code: Use case,Dependencies with yield: Early exit and `scope`
4,Response Status Code: Changing the default,Background Tasks: Introduction,Dependencies with yield: Early exit and `scope`,Middleware: Introduction
5,Return a Response Directly: Returning a custom...,Dependencies with yield: Early exit and `scope`,"Custom Response - HTML, Stream, File, others: ...",Advanced Dependencies: Dependencies with `yiel...


**After the change:** Background Tasks now appears fourth in the full-text results, and the cross-encoder hybrid ranks it **first**. Once the page is among the candidates, a model that reads the question and the passage together recognizes it as the answer. RRF doesn't: it merges the two lists by rank position alone, and the page appears in only one list, fourth, so pages that rank reasonably well in both lists stay ahead of it.

The lesson: retrieval decides what can be found, and reranking decides what comes first. A good reranker can't recover from a search that drops the words that matter.

A phrase search, written in double quotes, finds the page directly, but only because it uses the docs' own wording. Real users rarely know that wording, which is why the ranked methods above matter more.

In [15]:
top_hits(table.search('"after returning a response"', query_type="fts"))

['Background Tasks: Introduction']

## Step 7: Indexes for larger tables

So far, every vector search compared the question with all 814 rows. That's an exact, brute-force search, and at this size it takes milliseconds. A **vector index** groups similar vectors so a search checks only the most promising groups. That makes search much faster on large tables, at the cost of being approximate: it can occasionally miss a row that brute force would find.

The cell below builds an `IVF_PQ` index: IVF splits the vectors into partitions (`num_partitions`; each partition needs a few hundred vectors to train, so this small table gets only 3), and PQ compresses each vector into smaller codes (`num_sub_vectors`, which must divide the vector's 384 dimensions evenly). With a table this small, the index is here to show how it works, not because it's needed.

A **scalar index** speeds up filters in the same way. A bitmap index suits a column with only a few distinct values, such as `section`.

In [16]:
from lancedb.index import Bitmap, IvfPq

table.create_index("vector", config=IvfPq(distance_type="l2", num_partitions=3, num_sub_vectors=16), replace=True)
table.create_index("section", config=Bitmap(), replace=True)

for index in table.list_indices():
    print(f"{index.name}: {index.index_type} on {index.columns}")

section_idx: Bitmap on ['section']
text_idx: FTS on ['text']
vector_idx: IvfPq on ['vector']


Searches now use the vector index automatically. Two settings trade speed for accuracy: `nprobes` sets how many partitions to search (here, 2 of the 3), and `refine_factor` re-checks extra candidates against the full, uncompressed vectors. Compare the results with the exact search in Step 4.

In [17]:
(
    table.search("How do I make a query parameter optional?")
    .nprobes(2)
    .refine_factor(5)
    .limit(5)
    .select(["page_title", "heading", "_distance"])
    .to_pandas()
)

,page_title,heading,_distance
0,Query Parameters and String Validations,Default values,0.606595
1,Query Parameters,Optional parameters,0.705053
2,Query Parameters and String Validations,Required parameters,0.777589
3,Query Parameters and String Validations,Deprecating parameters,0.850110
4,Query Parameters,Required query parameters,0.869601


In the run saved in this notebook, the indexed search returned the same five results as the exact search in Step 4, with the same distances. The distances match because `refine_factor` re-scores the final candidates with the full vectors; the index only changes which candidates get considered. On a table of millions of rows, the same search would be far faster than checking every row.

## Step 8: Evolve the table and use versions

`add_columns` adds a column computed with a SQL expression, without rewriting the existing data. Here it estimates reading time at 200 words a minute.

Every change to a table, including adding rows, columns, or indexes, creates a new **version**. You can check out an earlier version to see the table as it was, then return to the latest.

In [18]:
version_before = table.version
table.add_columns({"reading_minutes": "CAST(word_count AS DOUBLE) / 200.0"})
print(f"Version {version_before} -> {table.version}")

table.search("How do I deploy with Docker?").limit(3).select(["page_title", "heading", "reading_minutes", "_distance"]).to_pandas()

Version 6 -> 7


,page_title,heading,reading_minutes,_distance
0,FastAPI in Containers - Docker,Deploy the Container Image,0.270,0.760900
1,FastAPI in Containers - Docker,Recap,0.470,0.812579
2,FastAPI in Containers - Docker,Build a Docker Image for FastAPI,0.095,0.869300


In [19]:
table.checkout(version_before)
print(f"Version {table.version} columns:", table.schema.names)

table.checkout_latest()
print(f"Version {table.version} columns:", table.schema.names)

Version 6 columns: ['id', 'page_title', 'heading', 'section', 'url', 'word_count', 'text', 'vector']
Version 7 columns: ['id', 'page_title', 'heading', 'section', 'url', 'word_count', 'text', 'vector', 'reading_minutes']


## Recap

This notebook downloaded and cleaned a real documentation set, stored it in LanceDB with automatic embeddings, and searched it four ways: vector, filtered, full-text, and hybrid with two rerankers. Only one combination answered the hardest question: a full-text index that keeps stop words, merged with vector search and reranked by a cross-encoder. It then added vector and scalar indexes, evolved the schema, and moved between table versions.

All data lives in the `data/` folder next to this notebook. Delete that folder to start over.